# Siamese — identity_mining_v2: frozen positives + safe negatives

**Standalone trên Kaggle:** chỉ upload notebook này; không cần package hoặc runner trong repo.

## Cách chạy
1. Gắn output `dataset_finetune_mining_v2` đã được duyệt từ notebook 04. Sửa `ROOT_FOLDER` và dùng `SAVE_DIR` mới.
2. Giữ `RUN_TRAINING=False`, chạy tất cả cell để kiểm tra bảng video → physical identity → split.
3. Quy tắc mặc định: `<identity>_0` và `<identity>_1` cùng vật thể. Kiểm tra toàn bộ mapping; tên ngoại lệ điền `IDENTITY_OVERRIDES`. Không tự gộp tên ngoài quy tắc.
4. Khi mapping/split đúng, đặt `RUN_TRAINING=True` và chạy lại. Dùng thư mục output riêng, không ghi đè checkpoint 0.700454.

## Thay đổi có chủ đích
- Split theo physical identity; negative pool chỉ từ video thuộc chính split.
- Nhánh negative từ positive crop khác phải có identity khác query.
- Train giữ uniform video sampling, positive cùng video và xác suất pool 0.7.
- Validation dùng mỗi positive đúng một lần trong danh sách triplet cố định; báo loss/accuracy theo sample.
- RNG Python/NumPy/torch/workers được seed. Không hứa bitwise determinism trên GPU.

## Giữ nguyên
MobileNetV3-Small, projection 576→256→128, normalization, augmentation, TripletMarginLoss margin 0.5,
AdamW lr 1e-4, weight decay 1e-4, cosine scheduler, batch 128, 15 epochs, AMP và train step.
Chưa đổi miner YOLOv8n, chưa lọc lại crop theo GT coverage, chưa sửa notebook 06. Không thêm DataParallel.

**Chưa có kết quả train/ST-IoU mới.** Val loss mới không so trực tiếp với val loss cũ vì split và triplets khác.
Nếu dataset chỉ còn một identity trong một split, notebook cảnh báo và dùng pool fallback;
nếu không có negative hợp lệ thì báo lỗi, tuyệt đối không dùng anchor làm negative.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from torch.cuda.amp import autocast, GradScaler
import cv2
import numpy as np
import os
import random
from glob import glob
from tqdm.notebook import tqdm
from PIL import Image
from pathlib import Path
import csv
import hashlib
import json
import math
import platform
import importlib.metadata
import re
from collections import Counter

# ================= PATHS: chỉ sửa tại đây =================
ROOT_FOLDER = '/kaggle/input/YOUR_MINING_V2_DATASET/dataset_finetune_mining_v2_run1'
SAVE_DIR = '/kaggle/working/siamese_identity_mining_v2'
EXPERIMENT_NAME = 'identity_mining_v2'

# Chạy False trước để xem mapping/split; xác nhận bảng rồi đổi True và chạy lại các cell.
RUN_TRAINING = False

# ================= TRAINING: giữ nguyên baseline =================
BATCH_SIZE = 128
LEARNING_RATE = 1e-4
NUM_EPOCHS = 15
MARGIN = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= IDENTITY / REPRODUCIBILITY =================
SPLIT_SEED = 42
TRAIN_SEED = 42
VAL_TRIPLET_SEED = 2026
VAL_FRACTION = 0.2
POOL_NEGATIVE_PROBABILITY = 0.7  # giữ tỷ lệ gốc 70% pool / 30% other identity

# Quy tắc đã được xác nhận: <identity>_0 và <identity>_1 là cùng vật thể.
# Giữ toàn bộ tiền tố, không dùng split('_')[0]. Tên khác quy tắc cần override.
IDENTITY_OVERRIDES = {
    # 'video_name_without_suffix': 'physical_object_id',
}
# None: shuffle identity bằng Python Random(SPLIT_SEED), lấy ceil(20%) cho val.
# Có thể ghi danh sách identity cụ thể để giữ cùng split qua các thí nghiệm.
VAL_IDENTITY_IDS = None

# Chỉ dùng khi negative cũ không theo format do notebook 04 sinh.
# Key: đường dẫn tương đối từ ROOT_FOLDER; value: video nguồn.
NEGATIVE_VIDEO_OVERRIDES = {
    # 'negatives/hard/custom_name.jpg': 'Backpack_0',
}

print(f"Device: {DEVICE} | Dataset: {ROOT_FOLDER}")
print(f"Experiment: {EXPERIMENT_NAME} | RUN_TRAINING = {RUN_TRAINING}")


In [ ]:
class DroneDegradation(object):
    """
    Class này mô phỏng chất lượng kém của ảnh drone:
    Thu nhỏ ảnh xuống cực bé rồi phóng to lại để tạo hiệu ứng vỡ hạt (pixelation).
    """
    def __init__(self, scale_range=(0.1, 0.3)): 
        self.scale_range = scale_range

    def __call__(self, img):
        w, h = img.size
        # Chọn ngẫu nhiên tỉ lệ thu nhỏ (ví dụ còn 20% kích thước thật)
        scale = random.uniform(*self.scale_range)
        new_w, new_h = max(16, int(w * scale)), max(16, int(h * scale))
        
        # 1. Thu nhỏ (Mất thông tin chi tiết)
        img_small = img.resize((new_w, new_h), resample=Image.BILINEAR)
        # 2. Phóng to lại kích thước cũ (Tạo răng cưa/mờ)
        img_back = img_small.resize((w, h), resample=Image.NEAREST)
        return img_back

def get_transforms(is_train=True):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    
    if is_train:
        # A. Anchor (Ảnh mẫu): PHẢI "PHÁ HỦY" ĐỂ GIỐNG ẢNH DRONE
        transform_anchor = transforms.Compose([
            transforms.Resize((224, 224)),
            
            # --- CẢI TIẾN QUAN TRỌNG NHẤT ---
            # 80% cơ hội bị làm vỡ hạt, mờ nhòe giống camera drone chất lượng thấp
            transforms.RandomApply([DroneDegradation(scale_range=(0.1, 0.3))], p=0.8),
            
            # Thêm rung lắc (Blur) mạnh
            transforms.RandomApply([
                transforms.GaussianBlur(kernel_size=(5, 9), sigma=(1.5, 3.0))
            ], p=0.6),
            
            # Sai lệch màu sắc (Color Jitter)
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
            
            # Các biến đổi hình học cơ bản
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(90), # Xoay mọi góc
            
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])
        
        # B. Drone Crop (Ảnh thật): Giữ nguyên hoặc augment nhẹ
        transform_drone = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(30),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])
        return transform_anchor, transform_drone
    
    else:
        # Val/Test
        transform_valid = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])
        return transform_valid, transform_valid


In [ ]:
# ================= INDEX / IDENTITY / SPLIT =================
def canonical_hash(value):
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':'))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def resolve_identity_map(video_ids, overrides=None):
    overrides = dict(overrides or {})
    unknown = set(overrides) - set(video_ids)
    if unknown:
        raise ValueError(f"Identity overrides có video không hợp lệ: {sorted(unknown)}")
    identities = {}
    for video_id in sorted(video_ids):
        if video_id in overrides:
            identity = overrides[video_id]
        else:
            match = re.fullmatch(r"(.+)_([01])", video_id)
            if match is None:
                raise ValueError(
                    f"Không suy identity cho '{video_id}'. Thêm IDENTITY_OVERRIDES, rồi kiểm tra mapping."
                )
            identity = match.group(1)
        if not isinstance(identity, str) or not identity.strip():
            raise ValueError(f"Identity rỗng/không hợp lệ cho {video_id}")
        identities[video_id] = identity
    return identities


def build_data_index(root_folder, negative_video_overrides=None):
    root = Path(root_folder).expanduser().resolve()
    for folder in ('anchors', 'positives', 'negatives'):
        if not (root / folder).is_dir():
            raise FileNotFoundError(f"Thiếu thư mục {root / folder}")
    anchors, positives = {}, {}
    for path in sorted((root / 'anchors').glob('*.jpg')):
        match = re.fullmatch(r"(.+)_ref_(\d+)\.jpg", path.name)
        if match is None:
            raise ValueError(f"Anchor không đúng format notebook 04: {path.name}")
        anchors.setdefault(match.group(1), []).append(path.relative_to(root).as_posix())
    for folder in sorted((root / 'positives').iterdir()):
        if folder.is_dir():
            positives[folder.name] = [
                path.relative_to(root).as_posix() for path in sorted(folder.glob('*.jpg'))
            ]
    all_video_ids = sorted(set(anchors) | set(positives))
    valid_video_ids = [
        video for video in all_video_ids if anchors.get(video) and positives.get(video)
    ]
    excluded = {
        video: {'anchors': len(anchors.get(video, [])), 'positives': len(positives.get(video, []))}
        for video in all_video_ids if video not in valid_video_ids
    }
    if not valid_video_ids:
        raise ValueError("Không có video có cả anchor và positive")
    overrides = dict(negative_video_overrides or {})
    used_overrides = set()
    negative_records = []
    errors = []
    for path in sorted((root / 'negatives').rglob('*.jpg')):
        relative = path.relative_to(root).as_posix()
        kind = path.relative_to(root / 'negatives').parts[0]
        if kind not in ('hard', 'background'):
            errors.append(f"{relative}: không thuộc hard/background")
            continue
        if relative in overrides:
            video_id = overrides[relative]
            used_overrides.add(relative)
            if not isinstance(video_id, str) or not video_id:
                errors.append(f"{relative}: override video nguồn không hợp lệ")
                continue
        else:
            # NB04: <video>_f<frame>_h<n>_<x>_<y>.jpg hoặc _bg<n>_...
            match = re.fullmatch(r"(.+)_f(\d+)_(h|bg)\d+_\d+_\d+\.jpg", path.name)
            if match is None or (match.group(3) == 'h') != (kind == 'hard'):
                errors.append(f"{relative}: không xác định được provenance từ format NB04")
                continue
            video_id = match.group(1)
        negative_records.append({'path': relative, 'video_id': video_id, 'kind': kind})
    if set(overrides) - used_overrides:
        errors.append(f"Negative overrides không dùng: {sorted(set(overrides)-used_overrides)}")
    if errors:
        raise ValueError("Không trộn negative không rõ video nguồn.\n" + "\n".join(errors[:20]) +
                         "\nSửa NEGATIVE_VIDEO_OVERRIDES hoặc kiểm tra tên file từ notebook 04.")
    return {
        'anchors': anchors, 'positives': positives, 'negatives': negative_records,
        'valid_video_ids': valid_video_ids, 'excluded_videos': excluded,
        # Hash này chỉ chứng thực danh sách đường dẫn/provenance, không hash nội dung ảnh.
        'schema_version': 1,
    }


def split_by_identity(identity_map, val_fraction=0.2, seed=42, val_identity_ids=None):
    identities = sorted(set(identity_map.values()))
    if len(identities) < 2:
        raise ValueError("Cần ít nhất 2 physical identities để tạo train/validation độc lập")
    if val_identity_ids is None:
        if isinstance(val_fraction, bool) or not isinstance(val_fraction, (int, float)) or not 0 < val_fraction < 1:
            raise ValueError("VAL_FRACTION phải nằm trong (0, 1)")
        shuffled = identities.copy()
        random.Random(seed).shuffle(shuffled)
        count = math.ceil(len(identities) * val_fraction)
        if count >= len(identities):
            raise ValueError("VAL_FRACTION để lại train rỗng")
        val_identities = sorted(shuffled[:count])
    else:
        val_identities = sorted(val_identity_ids)
        if (not val_identities or len(set(val_identities)) != len(val_identities)
                or set(val_identities) - set(identities) or len(val_identities) == len(identities)):
            raise ValueError("VAL_IDENTITY_IDS phải là tập con khác rỗng, không lặp, không lấy hết identities")
    train_identities = sorted(set(identities) - set(val_identities))
    train_videos = sorted(v for v, identity in identity_map.items() if identity in train_identities)
    val_videos = sorted(v for v, identity in identity_map.items() if identity in val_identities)
    return train_videos, val_videos, train_identities, val_identities


# ================= DATASET: train random, validation fixed =================
class SiameseDroneDataset(Dataset):
    def __init__(self, root_folder, data_index, identity_map, video_ids,
                 transform=None, is_train=True, val_seed=2026, pool_probability=0.7):
        self.root_folder = Path(root_folder).expanduser().resolve()
        self.is_train = is_train
        if not 0 <= pool_probability <= 1:
            raise ValueError("pool_probability phải nằm trong [0, 1]")
        self.pool_probability = pool_probability
        self.transform_anchor, self.transform_drone = transform if transform is not None else (None, None)
        self.identity_map = dict(identity_map)
        self.valid_video_ids = sorted(video_ids)
        if not self.valid_video_ids or len(set(self.valid_video_ids)) != len(self.valid_video_ids):
            raise ValueError("Split rỗng hoặc có video trùng")
        if set(self.valid_video_ids) - set(data_index['valid_video_ids']):
            raise ValueError("Split chứa video thiếu anchors/positives")
        if set(self.valid_video_ids) - set(self.identity_map):
            raise ValueError("Split chứa video chưa có physical identity")
        self.anchors_map = {v: data_index['anchors'][v] for v in self.valid_video_ids}
        self.positives_map = {v: data_index['positives'][v] for v in self.valid_video_ids}
        self.negative_records = [
            record for record in data_index['negatives'] if record['video_id'] in self.valid_video_ids
        ]
        self.negatives = [record['path'] for record in self.negative_records]
        self.negative_by_path = {record['path']: record for record in self.negative_records}
        self.other_video_ids = {
            v: [other for other in self.valid_video_ids if self.identity_map[other] != self.identity_map[v]]
            for v in self.valid_video_ids
        }
        for video_id in self.valid_video_ids:
            if not self.negatives and not self.other_video_ids[video_id]:
                raise ValueError(f"Không có negative hợp lệ cho {video_id}; không dùng anchor làm negative")
        self.positive_sets = {v: set(paths) for v, paths in self.positives_map.items()}
        self.anchor_sets = {v: set(paths) for v, paths in self.anchors_map.items()}
        self.fixed_triplets = None
        if not is_train:
            rng = random.Random(val_seed)  # không tiêu thụ RNG của training
            self.fixed_triplets = [
                self.sample_triplet(rng, video_id=video, positive_path=positive)
                for video in self.valid_video_ids for positive in self.positives_map[video]
            ]
            for triplet in self.fixed_triplets:
                self.validate_triplet(triplet)

    def __len__(self):
        return sum(len(self.positives_map[v]) for v in self.valid_video_ids)

    def sample_triplet(self, rng, video_id=None, positive_path=None):
        # Giữ train sampling uniform theo VIDEO như baseline; chưa đổi sang balanced identity batches.
        video_id = rng.choice(self.valid_video_ids) if video_id is None else video_id
        anchor = rng.choice(self.anchors_map[video_id])
        positive = rng.choice(self.positives_map[video_id]) if positive_path is None else positive_path
        other_ids = self.other_video_ids[video_id]
        use_pool = rng.random() < self.pool_probability and bool(self.negatives)
        if use_pool or not other_ids:
            if not self.negatives:
                raise ValueError(f"Không có negative hợp lệ cho {video_id}")
            negative = rng.choice(self.negative_records)
            neg_path, neg_video, kind = negative['path'], negative['video_id'], negative['kind']
            source = 'pool'
        else:
            neg_video = rng.choice(other_ids)
            neg_path = rng.choice(self.positives_map[neg_video])
            kind, source = 'positive_crop', 'other_identity'
        return {
            'anchor': anchor, 'positive': positive, 'negative': neg_path,
            'video_id': video_id, 'identity_id': self.identity_map[video_id],
            'negative_source': source, 'negative_kind': kind, 'negative_video_id': neg_video,
            # Với pool, đây là identity của video NGUỒN, không phải identity thực của crop background.
            'negative_source_identity': self.identity_map[neg_video],
        }

    def validate_triplet(self, triplet):
        video = triplet['video_id']
        neg_video = triplet['negative_video_id']
        if video not in self.valid_video_ids or neg_video not in self.valid_video_ids:
            raise ValueError("Triplet vượt ra ngoài split")
        if triplet['identity_id'] != self.identity_map[video]:
            raise ValueError("Query identity không đúng mapping")
        if triplet['anchor'] not in self.anchor_sets[video] or triplet['positive'] not in self.positive_sets[video]:
            raise ValueError("Anchor/positive không thuộc query video")
        if triplet['negative_source'] == 'other_identity':
            if self.identity_map[neg_video] == self.identity_map[video]:
                raise ValueError("Cùng physical identity bị lấy làm negative")
            if triplet['negative'] not in self.positive_sets[neg_video]:
                raise ValueError("Other-identity negative không thuộc video nguồn")
        elif triplet['negative_source'] == 'pool':
            record = self.negative_by_path.get(triplet['negative'])
            if record is None or record['video_id'] != neg_video:
                raise ValueError("Negative pool/provenance không thuộc split")
        else:
            raise ValueError("Loại negative không hợp lệ")
        if triplet['negative'] in (triplet['anchor'], triplet['positive']):
            raise ValueError("Cùng đường dẫn ảnh bị dùng làm positive và negative")

    def _load_image(self, path):
        img = cv2.imread(path)
        if img is None: return Image.new('RGB', (224, 224))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return Image.fromarray(img)

    def __getitem__(self, idx):
        triplet = self.sample_triplet(random) if self.is_train else self.fixed_triplets[idx]
        anchor_img = self._load_image(str(self.root_folder / triplet['anchor']))
        positive_img = self._load_image(str(self.root_folder / triplet['positive']))
        negative_img = self._load_image(str(self.root_folder / triplet['negative']))
        if self.transform_anchor:
            return (self.transform_anchor(anchor_img),
                    self.transform_drone(positive_img),
                    self.transform_drone(negative_img))
        return anchor_img, positive_img, negative_img


def split_paths(dataset):
    result = set(dataset.negatives)
    for video in dataset.valid_video_ids:
        result.update(dataset.anchors_map[video])
        result.update(dataset.positives_map[video])
    return result


def audit_splits(train_dataset, val_dataset, sample_count=1000):
    train_identities = {train_dataset.identity_map[v] for v in train_dataset.valid_video_ids}
    val_identities = {val_dataset.identity_map[v] for v in val_dataset.valid_video_ids}
    if train_identities & val_identities:
        raise ValueError("Physical identity giao giữa train và validation")
    if split_paths(train_dataset) & split_paths(val_dataset):
        raise ValueError("Đường dẫn ảnh giao giữa train và validation")
    counts = Counter()
    rng = random.Random(31415)  # audit không làm thay đổi train RNG
    for _ in range(sample_count):
        triplet = train_dataset.sample_triplet(rng)
        train_dataset.validate_triplet(triplet)
        counts[triplet['negative_source']] += 1
    for triplet in val_dataset.fixed_triplets:
        val_dataset.validate_triplet(triplet)
    return {
        'identity_overlap': [], 'path_overlap': [],
        'train_sampled_triplets_checked': sample_count, 'train_sampled_negative_sources': dict(counts),
        'validation_triplets_checked': len(val_dataset.fixed_triplets),
        'validation_negative_sources': dict(Counter(t['negative_source'] for t in val_dataset.fixed_triplets)),
        'same_identity_other_positive_negatives': 0,
        'note': 'Chỉ kiểm tra identity/provenance; chưa kiểm tra pixel crop chứa target (phần 2).',
    }


In [ ]:
class SiameseMobileNet(nn.Module):
    def __init__(self, pretrained=True, embedding_dim=576):
        super(SiameseMobileNet, self).__init__()
        if pretrained:
            full_model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        else:
            full_model = mobilenet_v3_small(weights=None)
        
        self.features = full_model.features
        
        # Projection Head
        self.projection = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128)
        )
        
    def forward_once(self, x):
        x = self.features(x)
        x = nn.functional.adaptive_avg_pool2d(x, (1, 1))
        x = x.flatten(1)
        x = self.projection(x)
        return torch.nn.functional.normalize(x, p=2, dim=1)
    
    def forward(self, anchor, positive, negative):
        return self.forward_once(anchor), self.forward_once(positive), self.forward_once(negative)


In [ ]:
class SiameseTrainer:
    def __init__(self, model, train_loader, val_loader, device, learning_rate, save_dir):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        
        # Optimizer: AdamW tốt hơn Adam thường
        self.optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
        
        # Scheduler: Cosine Annealing để hội tụ mượt mà
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=NUM_EPOCHS)
        
        # Loss: Margin 0.5 cho vector normalized
        self.criterion = nn.TripletMarginLoss(margin=MARGIN, p=2)
        self.scaler = GradScaler()        
        self.best_val_loss = float('inf')
    
    def calc_accuracy(self, a, p, n):
        # Tính khoảng cách Euclidean
        d_ap = (a - p).pow(2).sum(1)
        d_an = (a - n).pow(2).sum(1)
        # Accuracy: Tỉ lệ số lần khoảng cách AP < AN
        return (d_ap < d_an).float().mean()

    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0
        total_acc = 0
        
        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch} [Train]', leave=False)
        for anchor, positive, negative in pbar:
            anchor, positive, negative = anchor.to(self.device), positive.to(self.device), negative.to(self.device)
            
            self.optimizer.zero_grad()
            
            with autocast():
                emb_a, emb_p, emb_n = self.model(anchor, positive, negative)
                loss = self.criterion(emb_a, emb_p, emb_n)
                acc = self.calc_accuracy(emb_a, emb_p, emb_n)
            
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
            
            total_loss += loss.item()
            total_acc += acc.item()
            
            pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{acc.item():.2%}"})
        
        avg_loss = total_loss / len(self.train_loader)
        avg_acc = total_acc / len(self.train_loader)
        return avg_loss, avg_acc
    
    def validate(self):
        self.model.eval()
        total_loss = 0
        total_acc = 0
        total_samples = 0
        with torch.no_grad():
            for anchor, positive, negative in self.val_loader:
                anchor, positive, negative = anchor.to(self.device), positive.to(self.device), negative.to(self.device)
                with autocast():
                    emb_a, emb_p, emb_n = self.model(anchor, positive, negative)
                    loss = self.criterion(emb_a, emb_p, emb_n)
                    acc = self.calc_accuracy(emb_a, emb_p, emb_n)
                batch_samples = anchor.shape[0]
                total_loss += loss.item() * batch_samples
                total_acc += acc.item() * batch_samples
                total_samples += batch_samples
        if total_samples == 0:
            raise ValueError("Validation loader rỗng")
        # Mean theo sample, không cho batch cuối nhỏ trọng số ngang một batch đầy.
        return total_loss / total_samples, total_acc / total_samples

    def fit(self, epochs):
        print(f"Training {EXPERIMENT_NAME}: {epochs} epochs | Batch={BATCH_SIZE} | LR={LEARNING_RATE}")
        history = []
        best_epoch = None
        for epoch in range(1, epochs + 1):
            lr_used = self.optimizer.param_groups[0]['lr']
            train_loss, train_acc = self.train_epoch(epoch)
            val_loss, val_acc = self.validate()
            if not all(math.isfinite(v) for v in (train_loss, train_acc, val_loss, val_acc)):
                raise RuntimeError("Loss/accuracy không hữu hạn; dừng để tránh chọn checkpoint sai.")
            self.scheduler.step()
            curr_lr = self.optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch}: Train Loss={train_loss:.4f} (Acc={train_acc:.2%}) | "
                  f"Val Loss={val_loss:.4f} (Acc={val_acc:.2%}) | LR={curr_lr:.6f}")
            is_best = val_loss < self.best_val_loss
            if is_best:
                self.best_val_loss = val_loss
                best_epoch = epoch
                torch.save(self.model.state_dict(), self.save_dir / 'siamese_mobilenet_best.pth')
                print("  >>> Best model saved")
            row = {
                'epoch': epoch, 'train_loss': train_loss, 'train_accuracy': train_acc,
                'val_loss': val_loss, 'val_accuracy': val_acc,
                'lr_used': lr_used, 'lr_next': curr_lr, 'is_best': is_best,
            }
            history.append(row)
            with (self.save_dir / 'training_history.csv').open('w', newline='', encoding='utf-8') as handle:
                writer = csv.DictWriter(handle, fieldnames=list(row))
                writer.writeheader()
                writer.writerows(history)
            write_result_json(self.save_dir / 'training_history.json', history)
        checkpoint = self.save_dir / 'siamese_mobilenet_best.pth'
        report = {
            'status': 'completed', 'experiment': EXPERIMENT_NAME,
            'epochs_completed': len(history), 'best_epoch': best_epoch,
            'best_val_loss': self.best_val_loss,
            'best_val_accuracy': next(row['val_accuracy'] for row in history if row['epoch'] == best_epoch),
            'checkpoint': str(checkpoint),
            'checkpoint_sha256': hashlib.sha256(checkpoint.read_bytes()).hexdigest(),
            'validation_weighting': 'sample-weighted mean over fixed validation triplets',
            'local_st_iou': None,
            'note': 'Chưa chạy NB06. Không so trực tiếp val loss với split/triplets cũ.',
        }
        write_result_json(self.save_dir / 'run_summary.json', report)
        print("Training complete:", json.dumps(report, ensure_ascii=False))


In [ ]:
# ================= ARTIFACTS / DIAGNOSTICS =================
def write_locked_json(path, value):
    path = Path(path)
    if path.exists():
        with path.open(encoding='utf-8') as handle:
            existing = json.load(handle)
        if existing != value:
            raise ValueError(f"{path.name} đã tồn tại nhưng khác cấu hình/dữ liệu. Dùng SAVE_DIR mới.")
        return
    with path.open('x', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, allow_nan=False)


def write_result_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, allow_nan=False)
    temporary.replace(path)


def prepare_experiment():
    data_index = build_data_index(ROOT_FOLDER, NEGATIVE_VIDEO_OVERRIDES)
    identity_map = resolve_identity_map(data_index['valid_video_ids'], IDENTITY_OVERRIDES)
    train_videos, val_videos, train_ids, val_ids = split_by_identity(
        identity_map, VAL_FRACTION, SPLIT_SEED, VAL_IDENTITY_IDS,
    )
    train_dataset = SiameseDroneDataset(
        ROOT_FOLDER, data_index, identity_map, train_videos,
        get_transforms(True), is_train=True, pool_probability=POOL_NEGATIVE_PROBABILITY,
    )
    val_dataset = SiameseDroneDataset(
        ROOT_FOLDER, data_index, identity_map, val_videos,
        get_transforms(False), is_train=False, val_seed=VAL_TRIPLET_SEED,
        pool_probability=POOL_NEGATIVE_PROBABILITY,
    )
    integrity = audit_splits(train_dataset, val_dataset)
    negative_counts = Counter((r['video_id'], r['kind']) for r in data_index['negatives'])
    print("\nVIDEO -> PHYSICAL IDENTITY -> SPLIT")
    print(f"{'Video':26s} {'Identity':22s} {'Split':6s} {'Anchor':>7s} {'Pos':>7s} {'Hard':>7s} {'BG':>7s}")
    rows = []
    for video in data_index['valid_video_ids']:
        row = {
            'video_id': video, 'identity_id': identity_map[video],
            'split': 'train' if video in train_videos else 'val',
            'anchors': len(data_index['anchors'][video]), 'positives': len(data_index['positives'][video]),
            'hard_negatives': negative_counts[(video, 'hard')],
            'background_negatives': negative_counts[(video, 'background')],
        }
        rows.append(row)
        print(f"{video:26s} {identity_map[video]:22s} {row['split']:6s} {row['anchors']:7d} "
              f"{row['positives']:7d} {row['hard_negatives']:7d} {row['background_negatives']:7d}")
    excluded_negatives = [r for r in data_index['negatives'] if r['video_id'] not in identity_map]
    if data_index['excluded_videos']:
        print("WARN: video thiếu anchors/positives bị loại:", data_index['excluded_videos'])
    if excluded_negatives:
        print("WARN: negative không thuộc video có đủ dữ liệu bị loại khỏi cả hai split:",
              dict(Counter(r['video_id'] for r in excluded_negatives)))
    for label, dataset in (('TRAIN', train_dataset), ('VAL', val_dataset)):
        identities = sorted({identity_map[v] for v in dataset.valid_video_ids})
        print(f"{label}: {len(identities)} identities | {len(dataset.valid_video_ids)} videos | "
              f"{len(dataset)} positive samples | {len(dataset.negatives)} pool negatives")
        if len(identities) < 2:
            print(f"WARN {label}: chỉ có một identity; nhánh other-identity dùng pool fallback.")
        if not dataset.negatives:
            print(f"WARN {label}: không có pool negatives; dùng other-identity negatives.")
    print("Train identities:", train_ids)
    print("Val identities:", val_ids)
    print("Integrity:", json.dumps(integrity, ensure_ascii=False))
    print("Ví dụ validation triplets cố định (nguồn negative / identity video nguồn):")
    for triplet in val_dataset.fixed_triplets[:8]:
        print(triplet['video_id'], Path(triplet['positive']).name, '->',
              triplet['negative_source'], triplet['negative_source_identity'], Path(triplet['negative']).name)

    index_hash = canonical_hash(data_index)
    val_hash = canonical_hash(val_dataset.fixed_triplets)
    split_manifest = {
        'experiment': EXPERIMENT_NAME, 'root_folder': str(Path(ROOT_FOLDER).expanduser().resolve()),
        'identity_rule': 'terminal _0/_1 share the entire prefix; explicit overrides for exceptions',
        'identity_map': identity_map, 'identity_overrides': IDENTITY_OVERRIDES,
        'train_video_ids': train_videos, 'val_video_ids': val_videos,
        'train_identity_ids': train_ids, 'val_identity_ids': val_ids,
        'split_algorithm': 'sorted identities + Python Random(seed).shuffle + ceil(val_fraction*n)',
        'split_seed': SPLIT_SEED, 'val_fraction': VAL_FRACTION, 'explicit_val_identity_ids': VAL_IDENTITY_IDS,
        'train_seed': TRAIN_SEED, 'val_triplet_seed': VAL_TRIPLET_SEED,
        'pool_negative_probability': POOL_NEGATIVE_PROBABILITY,
        'data_index_sha256': index_hash, 'val_triplets_sha256': val_hash,
        'validation_policy': 'one fixed triplet per positive crop; deterministic valid transforms',
        'training_policy': 'random video sampling; positive from same video; other positives from different identity',
        'training_hyperparameters': {
            'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE, 'epochs': NUM_EPOCHS,
            'margin': MARGIN, 'optimizer': 'AdamW', 'weight_decay': 1e-4,
            'scheduler': 'CosineAnnealingLR', 'amp': True,
        },
    }
    summary = {
        'rows': rows, 'integrity': integrity, 'train_samples': len(train_dataset),
        'val_samples': len(val_dataset), 'train_pool_negatives': len(train_dataset.negatives),
        'val_pool_negatives': len(val_dataset.negatives),
        'excluded_videos': data_index['excluded_videos'],
        'excluded_negative_sources': dict(Counter(r['video_id'] for r in excluded_negatives)),
        'data_index_sha256': index_hash, 'val_triplets_sha256': val_hash,
        'index_hash_scope': 'relative paths/provenance only; image bytes are not hashed or validated',
    }
    save_dir = Path(SAVE_DIR)
    names = ('data_index.json', 'split_manifest.json', 'val_triplets.json', 'data_summary.json')
    if save_dir.exists() and any(save_dir.iterdir()) and not all((save_dir / name).is_file() for name in names):
        raise FileExistsError("SAVE_DIR không phải thư mục prepared của experiment này. Chọn thư mục mới.")
    save_dir.mkdir(parents=True, exist_ok=True)
    for name, value in zip(names, (data_index, split_manifest, val_dataset.fixed_triplets, summary)):
        write_locked_json(save_dir / name, value)
    print("Validation triplets SHA-256:", val_hash)
    print("Artifacts:", save_dir)
    return train_dataset, val_dataset, split_manifest


# ================= SEEDS / TRAIN EXECUTION =================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def main():
    seed_everything(TRAIN_SEED)
    train_dataset, val_dataset, split_manifest = prepare_experiment()
    if not RUN_TRAINING:
        print("\nCHƯA TRAIN. Kiểm tra mapping/split ở trên. Nếu đúng, đặt RUN_TRAINING=True và chạy lại.")
        return
    if len(train_dataset) < 2 or len(train_dataset) % BATCH_SIZE == 1:
        raise ValueError(
            "Train tạo batch cuối chỉ 1 ảnh, không hợp BatchNorm của projection head. "
            "Không tự đổi batch/drop_last. Gửi data_summary.json để chọn split phù hợp."
        )
    save_dir = Path(SAVE_DIR)
    if any((save_dir / name).exists() for name in ('siamese_mobilenet_best.pth', 'training_history.csv')):
        raise FileExistsError("Run đã có checkpoint/history. Dùng SAVE_DIR mới để không ghi đè kết quả.")
    train_generator = torch.Generator().manual_seed(TRAIN_SEED)
    val_generator = torch.Generator().manual_seed(VAL_TRIPLET_SEED)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
        worker_init_fn=seed_worker, generator=train_generator,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
        worker_init_fn=seed_worker, generator=val_generator,
    )
    environment = {
        'python': platform.python_version(), 'torch': torch.__version__,
        'torchvision': importlib.metadata.version('torchvision'),
        'opencv': cv2.__version__, 'numpy': np.__version__,
        'device': str(DEVICE), 'cuda': torch.version.cuda,
        'visible_gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        'cudnn_deterministic': torch.backends.cudnn.deterministic,
        'cudnn_benchmark': torch.backends.cudnn.benchmark,
        'note': 'Seeds controlled, but bitwise GPU determinism is not guaranteed. No DataParallel added.',
    }
    write_result_json(save_dir / 'environment.json', environment)
    write_result_json(save_dir / 'run_summary.json', {'status': 'training', 'experiment': EXPERIMENT_NAME})
    try:
        model = SiameseMobileNet(pretrained=True)
        trainer = SiameseTrainer(model, train_loader, val_loader, DEVICE, LEARNING_RATE, SAVE_DIR)
        trainer.fit(epochs=NUM_EPOCHS)
    except Exception as error:
        write_result_json(save_dir / 'run_summary.json', {
            'status': 'failed', 'experiment': EXPERIMENT_NAME,
            'error': f"{type(error).__name__}: {error}",
        })
        raise


if __name__ == "__main__":
    torch.cuda.empty_cache()
    main()


## Kết quả cần gửi lại sau khi chạy

**Trước training:** bảng mapping/split và `data_summary.json`. Nếu mapping sai, pool bất thường,
hoặc xuất hiện cảnh báo thiếu dữ liệu, sửa/xác nhận trước khi bật training.

**Sau training:** `training_history.csv`, `run_summary.json`, `split_manifest.json` và `environment.json`.
Giữ `val_triplets.json` để tái lập; file có thể lớn, chưa cần gửi toàn bộ nếu không có lỗi.

**Đánh giá:** dùng checkpoint mới `SAVE_DIR/siamese_mobilenet_best.pth` trong notebook 06 bằng cách chỉ đổi
`SIAMESE_MODEL_PATH`; giữ detector, TTA, reference scales, thresholds và temporal như baseline.
Gửi mean ST-IoU và điểm sáu video. Nếu dùng runner YAML, tạo config thử nghiệm riêng và cập nhật hash
checkpoint Siamese theo `run_summary.json`; không đổi YAML baseline.

Lưu ý: việc tách identity tạo tập train/validation mới, không phải ablation một tham số của run cũ.
So sánh điểm trên cùng public-test tự gán nhãn; không khẳng định mức tăng trước khi chạy.
Các negative pool vẫn có thể chứa target do bộ lọc IoU của notebook 04 — vấn đề đó dành cho phần 2.
